# Crawl Hanzii, giữ CVDICT làm nguồn gốc

Notebook này tạo các file CSV trung gian trước khi import vào PostgreSQL. CVDICT là dữ liệu gốc; dữ liệu Hanzii chỉ được lưu như phần bổ sung.

Chạy thử một batch nhỏ trước. Chỉ chạy toàn bộ sau khi đã xác nhận quyền sử dụng dữ liệu Hanzii và kiểm tra parser.

In [1]:
from __future__ import annotations

import csv
import html
import json
import re
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import quote
from urllib.request import Request, urlopen
from urllib.robotparser import RobotFileParser

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CVDICT_PATH = PROJECT_ROOT / 'backend' / 'app' / 'data' / 'CVDICT.u8'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'hanzii_crawl'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Chạy thử trước. Đặt None sau khi parser đã được kiểm tra.
MAX_WORDS = 20
REQUEST_DELAY_SECONDS = 1.5
USER_AGENT = 'MandarinFlowDictionaryImporter/1.0 (+https://mandarinflow.online)'
HANZII_URL = 'https://hanzii.net/search/word/{word}?hl=vi'


In [2]:
CVDICT_PATTERN = re.compile(r'^(?P<traditional>\S+)\s+(?P<simplified>\S+)\s+\[(?P<pinyin>[^\]]+)\]\s+/((?P<meaning>.+))/\s*$')

def clean_meaning(value: str) -> str:
    parts = []
    for item in value.split('/'):
        item = item.strip()
        if item and not item.startswith('LT:'):
            parts.append(item)
    return '; '.join(parts)

def read_cvdict(path: Path) -> list[dict[str, str]]:
    rows = []
    seen = set()
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        match = CVDICT_PATTERN.match(line)
        if not match:
            continue
        row = {
            'traditional': match.group('traditional'),
            'simplified': match.group('simplified'),
            'pinyin_numbered': match.group('pinyin'),
            'cvdict_meaning': clean_meaning(match.group('meaning')),
        }
        key = (row['simplified'], row['traditional'], row['pinyin_numbered'])
        if key not in seen:
            rows.append(row)
            seen.add(key)
    return rows

cvdict_rows = read_cvdict(CVDICT_PATH)
len(cvdict_rows), cvdict_rows[:3]

(122595,
 [{'traditional': '%',
   'simplified': '%',
   'pinyin_numbered': 'pa1',
   'cvdict_meaning': 'phần trăm (Đài Loan)'},
  {'traditional': '2019冠狀病毒病',
   'simplified': '2019冠状病毒病',
   'pinyin_numbered': 'er4 ling2 yi1 jiu3 guan1 zhuang4 bing4 du2 bing4',
   'cvdict_meaning': 'COVID-19, bệnh coronavirus được xác định năm 2019'},
  {'traditional': '21三體綜合症',
   'simplified': '21三体综合症',
   'pinyin_numbered': 'er4 shi2 yi1 san1 ti3 zong1 he2 zheng4',
   'cvdict_meaning': 'bệnh tam nhiễm sắc thể; hội chứng Down'}])

In [3]:
def write_csv(path: Path, rows: list[dict[str, str]], fieldnames: list[str]) -> None:
    with path.open('w', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

base_fields = ['traditional', 'simplified', 'pinyin_numbered', 'cvdict_meaning']
write_csv(OUTPUT_DIR / 'cvdict_base.csv', cvdict_rows, base_fields)
print('Đã xuất', len(cvdict_rows), 'dòng:', OUTPUT_DIR / 'cvdict_base.csv')

Đã xuất 122595 dòng: /home/datnguyen/Documents/GIT_PROJECT/learning_chinese_through_vid/youtube-language-learning/data/hanzii_crawl/cvdict_base.csv


In [4]:
def can_fetch(url: str) -> bool:
    robots_url = 'https://hanzii.net/robots.txt'
    parser = RobotFileParser()
    parser.set_url(robots_url)
    try:
        parser.read()
        return parser.can_fetch(USER_AGENT, url)
    except (HTTPError, URLError, TimeoutError) as exc:
        raise RuntimeError(f'Không đọc được robots.txt của Hanzii: {exc}') from exc

def fetch_html(word: str) -> str:
    url = HANZII_URL.format(word=quote(word))
    if not can_fetch(url):
        raise PermissionError(f'robots.txt không cho phép crawl: {url}')
    request = Request(url, headers={'User-Agent': USER_AGENT, 'Accept-Language': 'vi,en;q=0.8'})
    with urlopen(request, timeout=25) as response:
        return response.read().decode('utf-8', errors='replace')

def visible_text(raw_html: str) -> str:
    text = re.sub(r'<script\b[^>]*>.*?</script>', ' ', raw_html, flags=re.I | re.S)
    text = re.sub(r'<style\b[^>]*>.*?</style>', ' ', text, flags=re.I | re.S)
    text = re.sub(r'<[^>]+>', ' ', text)
    return re.sub(r'\s+', ' ', html.unescape(text)).strip()

def parse_hanzii(word: str, raw_html: str) -> dict[str, str]:
    # Hanzii có thể thay đổi markup; giữ raw_text để không mất dữ liệu khi cần điều chỉnh parser.
    text = visible_text(raw_html)
    pinyin_match = re.search(r'\b[a-züv]+[1-5](?:\s+[a-züv]+[1-5])*\b', text, re.I)
    return {
        'word': word,
        'hanzii_pinyin': pinyin_match.group(0) if pinyin_match else '',
        'hanzii_meaning': '',
        'hanzii_part_of_speech': '',
        'hanzii_collocations_json': '[]',
        'hanzii_examples_json': '[]',
        'raw_text': text[:12000],
        'status': 'needs_parser_review',
    }


In [5]:
# Crawl thử một batch nhỏ và có thể chạy tiếp nhờ crawl_errors.csv.
sample = cvdict_rows[:MAX_WORDS] if MAX_WORDS is not None else cvdict_rows
enrichment_rows = []
error_rows = []

for index, source_row in enumerate(sample, start=1):
    word = source_row['simplified']
    try:
        raw_html = fetch_html(word)
        enrichment_rows.append(parse_hanzii(word, raw_html))
        print(f'[{index}/{len(sample)}] OK {word}')
    except Exception as exc:
        error_rows.append({'word': word, 'error_type': type(exc).__name__, 'error': str(exc)})
        print(f'[{index}/{len(sample)}] ERROR {word}: {exc}')
    time.sleep(REQUEST_DELAY_SECONDS)

enrichment_fields = ['word', 'hanzii_pinyin', 'hanzii_meaning', 'hanzii_part_of_speech', 'hanzii_collocations_json', 'hanzii_examples_json', 'raw_text', 'status']
write_csv(OUTPUT_DIR / 'hanzii_enrichment.csv', enrichment_rows, enrichment_fields)
write_csv(OUTPUT_DIR / 'crawl_errors.csv', error_rows, ['word', 'error_type', 'error'])
print(f'Đã crawl {len(enrichment_rows)} dòng, lỗi {len(error_rows)} dòng')

[1/20] OK %
[2/20] OK 2019冠状病毒病
[3/20] OK 21三体综合症
[4/20] OK 3C
[5/20] OK 3P
[6/20] OK 3Q
[7/20] OK 421
[8/20] OK 502胶
[9/20] OK 88
[10/20] OK 996
[11/20] OK A
[12/20] OK AA制
[13/20] OK AB制
[14/20] OK ACG
[15/20] OK A咖
[16/20] OK A圈儿
[17/20] OK A片
[18/20] OK A菜
[19/20] OK A货
[20/20] OK B
Đã crawl 20 dòng, lỗi 0 dòng


In [6]:
# Ghép CVDICT và Hanzii; chỉ dùng các cột đã kiểm tra.
hanzii_by_word = {row['word']: row for row in enrichment_rows}
merged = []
for row in cvdict_rows:
    extra = hanzii_by_word.get(row['simplified'], {})
    merged.append({
        **row,
        'hanzii_pinyin': extra.get('hanzii_pinyin', ''),
        'hanzii_meaning': extra.get('hanzii_meaning', ''),
        'hanzii_part_of_speech': extra.get('hanzii_part_of_speech', ''),
        'hanzii_collocations_json': extra.get('hanzii_collocations_json', '[]'),
        'hanzii_examples_json': extra.get('hanzii_examples_json', '[]'),
        'hanzii_status': extra.get('status', 'not_crawled'),
    })

merged_fields = base_fields + ['hanzii_pinyin', 'hanzii_meaning', 'hanzii_part_of_speech', 'hanzii_collocations_json', 'hanzii_examples_json', 'hanzii_status']
write_csv(OUTPUT_DIR / 'dictionary_merged.csv', merged, merged_fields)
print('Đã xuất file hợp nhất:', OUTPUT_DIR / 'dictionary_merged.csv')

Đã xuất file hợp nhất: /home/datnguyen/Documents/GIT_PROJECT/learning_chinese_through_vid/youtube-language-learning/data/hanzii_crawl/dictionary_merged.csv


## Kiểm tra trước khi chạy toàn bộ

1. Mở `hanzii_enrichment.csv` và kiểm tra `raw_text`.
2. Hoàn thiện `parse_hanzii()` theo markup/API thực tế của Hanzii.
3. Đổi `MAX_WORDS = None` để crawl toàn bộ CVDICT.
4. Chỉ sau khi kiểm tra CSV mới viết bước import PostgreSQL.